# Popularity Baseline trong bài toán gợi ý phim với phản hồi ẩn

## Tóm tắt (Abstract)
Notebook này xây dựng mô hình cơ sở dựa trên độ phổ biến (Popularity Baseline) cho bài toán gợi ý phim. Phương pháp xếp hạng item theo số lượng tương tác dương quan sát được trong tập huấn luyện, với tương tác dương được định nghĩa bởi $r_{u,i} \geq \tau$, trong đó $\tau$ là `POSITIVE_THRESHOLD`. Thiết lập Leave-One-Out (LOO) được sử dụng để đánh giá khả năng gợi ý ở mức người dùng.

## 1. Giới thiệu (Introduction)
Popularity Baseline là mốc so sánh không học tham số, phản ánh giả thuyết rằng các item được nhiều người dùng đánh giá tích cực có xác suất được đề xuất cao hơn. Mặc dù không khai thác đặc trưng cá nhân hóa, mô hình này hữu ích để kiểm tra xem các phương pháp phức tạp hơn có thực sự vượt qua tín hiệu phổ biến toàn cục hay không.

| Thành phần | Mô tả |
|---|---|
| Thiết lập chia dữ liệu | Leave-One-Out theo từng người dùng |
| Tín hiệu huấn luyện | Phản hồi ẩn từ rating thỏa $r_{u,i} \geq \tau$ |
| Chiến lược xếp hạng | Sắp xếp item theo số tương tác dương trong tập huấn luyện |
| Nguyên tắc đánh giá | Loại bỏ item đã xuất hiện trong lịch sử huấn luyện của người dùng |


## 2. Phương pháp nghiên cứu (Methodology)

### 2.1 Thiết lập môi trường và cấu hình
Cell cấu hình khai báo nguồn dữ liệu, ngưỡng phản hồi ẩn $\tau$, số lượng rating tối thiểu trên mỗi người dùng và danh sách giá trị $K$ dùng trong đánh giá Top-$K$. Các tham số này xác định toàn bộ giao thức thực nghiệm của baseline độ phổ biến.


In [43]:
# [Giải thích cell]
# - Mục đích: nạp thư viện và khai báo toàn bộ cấu hình thực nghiệm của notebook.
# - Đầu vào chính: URL dữ liệu, thư mục checkpoint, ngưỡng phản hồi ẩn và siêu tham số mô hình.
# - Kết quả tạo ra: các hằng số cấu hình được các cell phía sau sử dụng thống nhất.
# - Lưu ý: cấu hình thuộc biến thể Popularity Baseline, do đó cần giữ cố định khi so sánh với notebook khác.

import os
import warnings
from collections import Counter

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Nguồn dữ liệu ────────────────────────────────────────────────────────────────
MOVIES_URL  = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/movies.csv"
RATINGS_URL = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/ratings.csv"
USERS_URL   = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/user_profiles.csv"

# ── Cấu hình thực nghiệm ────────────────────────────────────────────────────────────────
POSITIVE_THRESHOLD = 7     # Giống LightFM Baseline
MIN_RATINGS        = 10    # Giống LightFM Baseline
K_VALUES           = (5, 10, 20, 50)

print("[Popularity Baseline] Cấu hình:")
print(f"  positive_threshold : {POSITIVE_THRESHOLD}")
print(f"  min_ratings_filter : {MIN_RATINGS}")
print(f"  K values           : {K_VALUES}")
print(f"  Split              : Leave-One-Out (LOO) per user")
print(f"  Strategy           : Global popularity (train positive count)")

[Popularity Baseline] Config:
  positive_threshold : 7
  min_ratings_filter : 10
  K values           : (5, 10, 20, 50)
  Split              : Leave-One-Out (LOO) per user
  Strategy           : Global popularity (train positive count)


### 2.2 Các hàm hỗ trợ
Các hàm hỗ trợ chuẩn hóa chuỗi, tách token và xử lý giá trị thiếu nhằm bảo đảm dữ liệu đầu vào có định dạng nhất quán. Các bước này không học tham số từ tập validation hoặc test.


In [44]:
# [Giải thích cell]
# - Mục đích: định nghĩa các hàm tiền xử lý văn bản và token dùng lại trong notebook.
# - Đầu vào chính: các cột metadata dạng chuỗi, thường chứa token phân tách bằng dấu `|`.
# - Kết quả tạo ra: chuỗi đã làm sạch, danh sách token và tập token đủ phổ biến để làm đặc trưng.
# - Lưu ý: các hàm này chỉ chuẩn hóa dữ liệu; không học tham số từ validation hoặc test.

def cleanup_column(series):
    """Fix encoding artefacts in pipe-separated string Series."""
    replacements = {
        '| ': '|', '\xc3\xa9': 'é', '\xc3\x81': 'Á', '\xc3\x93': 'Ó',
        '\xc3\xa1': 'á', '\xc3\xb3': 'ó', '\xc3\xb1': 'ñ', '\xc3\xad': 'í',
        '\xc3\xba': 'ú', 'Ã©': 'é', 'Ã¡': 'á', 'Ã³': 'ó', 'Ã±': 'ñ',
        'Ãº': 'ú',
    }
    col = series.copy()
    for bad, good in replacements.items():
        col = col.str.replace(bad, good, regex=False)
    return col


print("Helpers defined: cleanup_column")

Helpers defined: cleanup_column


### 2.3 Nạp dữ liệu
Ba bảng dữ liệu chính gồm `movies.csv`, `ratings.csv` và `user_profiles.csv` được nạp vào bộ nhớ. Cột thời gian được chuẩn hóa để phục vụ chiến lược Leave-One-Out theo thứ tự thời gian.


In [45]:
# [Giải thích cell]
# - Mục đích: nạp ba bảng dữ liệu gốc gồm rating, hồ sơ người dùng và metadata phim.
# - Đầu vào chính: các đường dẫn `RATINGS_URL`, `USERS_URL` và `MOVIES_URL`.
# - Kết quả tạo ra: `ratings_df`, `users_df`, `movies_df` và cột thời gian đã được chuẩn hóa.
# - Lưu ý: kiểm tra kích thước bảng sau khi nạp giúp phát hiện sớm lỗi đọc dữ liệu.

ratings_df = pd.read_csv(RATINGS_URL)
users_df   = pd.read_csv(USERS_URL)
movies_df  = pd.read_csv(MOVIES_URL)

ratings_df["date"] = pd.to_datetime(ratings_df["date"])

print(f"Users   : {users_df['id'].nunique()}")
print(f"Movies  : {movies_df['id'].nunique()}")
print(f"Ratings : {len(ratings_df):,}")
print(f"Rating range: {ratings_df['rate'].min()} – {ratings_df['rate'].max()}")
print(f"Date range  : {ratings_df['date'].min().date()} – {ratings_df['date'].max().date()}")

Users   : 482
Movies  : 78628
Ratings : 1,172,038
Rating range: 1 – 10
Date range  : 2002-08-14 – 2021-04-01


### 2.4 Lọc dữ liệu hợp lệ
Các bản ghi không khớp giữa bảng rating, bảng phim và hồ sơ người dùng được loại bỏ. Đồng thời, chỉ giữ người dùng có ít nhất `MIN_RATINGS` rating để bảo đảm mỗi người dùng có đủ dữ liệu cho train, validation và test.


In [46]:
# [Giải thích cell]
# - Mục đích: thực hiện một bước xử lý trong pipeline thực nghiệm của notebook.
# - Đầu vào chính: các biến đã được tạo ở những cell trước.
# - Kết quả tạo ra: biến trung gian hoặc bảng kết quả dùng cho các bước tiếp theo.
# - Lưu ý: comment này được bổ sung để giúp người đọc theo dõi luồng xử lý mà không cần chạy lại code.

# 2.1 Lọc orphan users (có rating nhưng KHÔNG có profile)
users_in_profile = set(users_df["id"].unique())
ratings_df = ratings_df[ratings_df["id"].isin(users_in_profile)].copy()
print(f"Sau lọc orphan users    : {len(ratings_df):,} ratings")

# 2.2 Lọc user có ít hơn MIN_RATINGS ratings
rating_counts = ratings_df.groupby("id")["movie_id"].count()
valid_users   = rating_counts[rating_counts >= MIN_RATINGS].index
ratings_df    = ratings_df[ratings_df["id"].isin(valid_users)].copy()
users_df      = users_df[users_df["id"].isin(valid_users)].copy()
print(f"Sau lọc min {MIN_RATINGS} ratings : {len(ratings_df):,} ratings, {ratings_df['id'].nunique()} users")

# 2.3 Lọc movie không tồn tại trong movies.csv
movies_in_table = set(movies_df["id"].unique())
ratings_df = ratings_df[ratings_df["movie_id"].isin(movies_in_table)].copy()
print(f"Sau lọc missing movies  : {len(ratings_df):,} ratings")

Sau lọc orphan users    : 963,266 ratings
Sau lọc min 10 ratings : 963,231 ratings, 474 users
Sau lọc missing movies  : 963,228 ratings


### 2.5 Chuẩn hóa văn bản metadata
Các cột văn bản được làm sạch để thống nhất encoding và định dạng token. Trong baseline độ phổ biến, metadata không tham gia trực tiếp vào mô hình nhưng vẫn được chuẩn hóa để phục vụ phần minh họa gợi ý.


In [47]:
# [Giải thích cell]
# - Mục đích: thực hiện một bước xử lý trong pipeline thực nghiệm của notebook.
# - Đầu vào chính: các biến đã được tạo ở những cell trước.
# - Kết quả tạo ra: biến trung gian hoặc bảng kết quả dùng cho các bước tiếp theo.
# - Lưu ý: comment này được bổ sung để giúp người đọc theo dõi luồng xử lý mà không cần chạy lại code.

for col in ["country_name", "genres", "topics"]:
    if col in movies_df.columns:
        movies_df[col] = cleanup_column(movies_df[col])

print("Text cleanup hoàn tất.")

Text cleanup hoàn tất.


### 2.6 Chia dữ liệu Leave-One-Out và áp dụng threshold
Quy trình dữ liệu được tách thành hai giai đoạn rõ ràng:

1. **Chia Leave-One-Out trên rating gốc:** `ratings_df` sau khi lọc hợp lệ vẫn giữ đầy đủ giá trị rating. Với mỗi người dùng, rating được sắp xếp theo thời gian; rating mới nhất được đưa vào `test_df`, rating liền trước được đưa vào `valid_df`, và toàn bộ rating còn lại được đưa vào `train_df`.
2. **Áp dụng threshold sau khi đã chia:** các split `train_df`, `valid_df` và `test_df` vẫn là dữ liệu rating gốc. Khi xây dựng tương tác phản hồi ẩn, chỉ các dòng thỏa $r_{u,i} \geq \tau$ mới được chuyển thành tương tác dương.

Điều này có nghĩa là threshold không được dùng để quyết định dòng nào thuộc train, validation hay test. Threshold chỉ quyết định dòng nào trở thành tương tác dương khi huấn luyện hoặc đánh giá. Cách làm này giữ đúng thứ tự thời gian của hành vi người dùng và tránh làm thay đổi tương tác mới nhất trước khi chia dữ liệu.


In [48]:
# [Giải thích cell]
# - Mục đích: chia dữ liệu Leave-One-Out trên rating gốc, trước khi áp dụng threshold phản hồi ẩn.
# - Cách làm: sắp xếp rating theo thời gian cho từng user; lấy rating mới nhất làm test, rating liền trước làm validation, phần còn lại làm train.
# - Kết quả tạo ra: `train_df`, `valid_df`, `test_df`; các bảng này vẫn giữ rating gốc, gồm cả rating cao và thấp.
# - Lưu ý: threshold `rate >= POSITIVE_THRESHOLD` chỉ được áp dụng ở các bước xây dựng interaction/evaluation sau đó.

ratings_df = ratings_df.sort_values(["id", "date"]).reset_index(drop=True)

train_idx = []
valid_idx = []
test_idx  = []

for uid, group in ratings_df.groupby("id", sort=False):
    idx = group.index.tolist()
    if len(idx) < 3:
        train_idx.extend(idx)
        continue
    test_idx.append(idx[-1])
    valid_idx.append(idx[-2])
    train_idx.extend(idx[:-2])

train_df = ratings_df.loc[train_idx].copy().reset_index(drop=True)
valid_df = ratings_df.loc[valid_idx].copy().reset_index(drop=True)
test_df  = ratings_df.loc[test_idx].copy().reset_index(drop=True)

print(f"Train : {len(train_df):>7,} ratings  |  {train_df['id'].nunique()} users")
print(f"Valid : {len(valid_df):>7,} ratings  |  {valid_df['id'].nunique()} users  (1 rating/user)")
print(f"Test  : {len(test_df):>7,} ratings  |  {test_df['id'].nunique()} users  (1 rating/user)")
print(f"\nDate range train: {train_df['date'].min().date()} – {train_df['date'].max().date()}")
print(f"Date range valid: {valid_df['date'].min().date()} – {valid_df['date'].max().date()}")
print(f"Date range test : {test_df['date'].min().date()} – {test_df['date'].max().date()}")

Train : 962,280 ratings  |  474 users
Valid :     474 ratings  |  474 users  (1 rating/user)
Test  :     474 ratings  |  474 users  (1 rating/user)

Date range train: 2002-08-14 – 2021-03-31
Date range valid: 2011-12-06 – 2021-04-01
Date range test : 2011-12-08 – 2021-04-01


### 2.7 Phân loại item warm/cold
Item được xem là warm nếu đã xuất hiện trong tập huấn luyện và cold nếu chưa xuất hiện trong train. Phân tách này cho phép đánh giá riêng mức độ phụ thuộc của baseline vào độ phổ biến quan sát được.


In [49]:
# [Giải thích cell]
# - Mục đích: tách item trong tập kiểm thử thành warm và cold theo sự xuất hiện trong train.
# - Đầu vào chính: `train_df` và `test_df`.
# - Kết quả tạo ra: `test_warm_df`, `test_cold_df` cùng các phiên bản positive nếu có.
# - Lưu ý: phân tách này giúp đánh giá riêng khả năng xử lý item cold-start.

movies_in_train = set(train_df["movie_id"].unique())

test_warm_mask = test_df["movie_id"].isin(movies_in_train)
test_cold_mask = ~test_warm_mask

test_warm_df = test_df[test_warm_mask].copy().reset_index(drop=True)
test_cold_df = test_df[test_cold_mask].copy().reset_index(drop=True)

print(f"Unique items in train : {len(movies_in_train):,}")
print(f"Unique items in valid : {valid_df['movie_id'].nunique():,}")
print(f"Unique items in test  : {test_df['movie_id'].nunique():,}")
print()
print(f"TEST total : {len(test_df):>6,} ratings")
print(f"TEST warm  : {len(test_warm_df):>6,} ratings  "
      f"({len(test_warm_df)/len(test_df)*100:.1f}%)  "
      f"— {test_warm_df['movie_id'].nunique():,} unique movies")
print(f"TEST cold  : {len(test_cold_df):>6,} ratings  "
      f"({len(test_cold_df)/len(test_df)*100:.1f}%)  "
      f"— {test_cold_df['movie_id'].nunique():,} unique movies")

Unique items in train : 75,115
Unique items in valid : 398
Unique items in test  : 407

TEST total :    474 ratings
TEST warm  :    446 ratings  (94.1%)  — 379 unique movies
TEST cold  :     28 ratings  (5.9%)  — 28 unique movies


### 2.8 Tính điểm phổ biến
Điểm phổ biến của item $i$ được định nghĩa là số lượng tương tác dương trong tập huấn luyện:

$$
\text{pop}(i) = \sum_u \mathbb{1}(r_{u,i} \geq \tau).
$$

Chỉ dữ liệu train được sử dụng trong công thức này để tránh rò rỉ dữ liệu từ validation và test.


In [50]:
# [Giải thích cell]
# - Mục đích: tính độ phổ biến item từ tập huấn luyện sau threshold.
# - Đầu vào train model: `train_df` được lọc thành `train_positives` với `rate >= POSITIVE_THRESHOLD`.
# - Kết quả tạo ra: `popularity_counts`, `popularity_rank` và `global_pop_list`.
# - Lưu ý: validation/test không được dùng để tính độ phổ biến.

# Tính popularity: đếm positive interactions trong train
train_positives = train_df[train_df["rate"] >= POSITIVE_THRESHOLD]

popularity_counts = (
    train_positives
    .groupby("movie_id")["id"]
    .count()
    .rename("pop_count")
    .reset_index()
    .sort_values("pop_count", ascending=False)
    .reset_index(drop=True)
)

# Đảm bảo tất cả movies trong dataset đều có entry (cold items = 0)
all_movie_ids = pd.DataFrame({"movie_id": movies_df["id"].tolist()})
popularity_counts = all_movie_ids.merge(popularity_counts, on="movie_id", how="left")
popularity_counts["pop_count"] = popularity_counts["pop_count"].fillna(0).astype(int)
popularity_counts = popularity_counts.sort_values("pop_count", ascending=False).reset_index(drop=True)

# Tạo mapping: movie_id → rank (0 = most popular)
popularity_rank   = {row["movie_id"]: rank for rank, row in popularity_counts.iterrows()}
popularity_score  = {row["movie_id"]: row["pop_count"] for _, row in popularity_counts.iterrows()}

# Danh sách movie_ids đã sort theo popularity (dùng cho recommendation)
global_pop_list = popularity_counts["movie_id"].tolist()  # index 0 = most popular

n_items_with_pop  = (popularity_counts["pop_count"] > 0).sum()
n_items_cold_pop  = (popularity_counts["pop_count"] == 0).sum()

print(f"[Popularity Baseline] Train positive threshold : rate >= {POSITIVE_THRESHOLD}")
print(f"  Total train positives          : {len(train_positives):,}")
print(f"  Items có pop_count > 0         : {n_items_with_pop:,}")
print(f"  Items với pop_count = 0 (cold) : {n_items_cold_pop:,}")
print(f"  Max pop_count  : {popularity_counts['pop_count'].max()}")
print(f"  Mean pop_count : {popularity_counts['pop_count'].mean():.2f}")
print(f"\nTop 10 most popular items:")
display(
    popularity_counts.head(10).merge(
        movies_df[["id", "original_title", "genres"]].rename(columns={"id": "movie_id"}),
        on="movie_id", how="left"
    )[["movie_id", "original_title", "genres", "pop_count"]]
)

[Popularity Baseline] Train positive threshold : rate >= 7
  Total train positives          : 412,438
  Items có pop_count > 0         : 36,478
  Items với pop_count = 0 (cold) : 42,150
  Max pop_count  : 386
  Mean pop_count : 5.25

Top 10 most popular items:


,movie_id,original_title,genres,pop_count
0,160882,Pulp Fiction,Thriller,386
1,809297,The Godfather,Drama,366
2,575149,Seven (Se7en),Thriller|Intriga,366
3,867354,The Dark Knight,Thriller|Acción|Drama,361
4,656153,Schindler's List,Drama,353
5,971380,Inception,Ciencia ficción|Thriller|Intriga|Acción,350
6,161026,The Shawshank Redemption,Drama,349
7,539054,Gran Torino,Drama,349
8,746997,Inglourious Basterds,Bélico|Acción|Comedia,346
9,505307,American Beauty,Drama|Comedia,345


### 2.9 Hàm sinh gợi ý
Danh sách gợi ý được tạo bằng cách sắp xếp item theo $\text{pop}(i)$ giảm dần và loại bỏ các item đã xuất hiện trong lịch sử huấn luyện của người dùng. Vì mô hình không cá nhân hóa, mọi người dùng chia sẻ cùng thứ tự phổ biến toàn cục trước bước mask lịch sử.


In [51]:
# [Giải thích cell]
# - Mục đích: sinh danh sách gợi ý theo thứ tự phổ biến toàn cục.
# - Đầu vào chính: user_id, lịch sử train và danh sách phim đã xếp hạng theo popularity.
# - Kết quả tạo ra: top-N movie_id sau khi loại bỏ phim người dùng đã tương tác.
# - Lưu ý: mô hình này không cá nhân hóa ngoài bước mask lịch sử.

def get_pop_recommendations(user_id, train_df, global_pop_list, n_recs=50):
    """
    Trả về list top-n_recs movie_ids theo popularity,
    loại trừ các items user đã tương tác trong train.
    """
    seen = set(train_df[train_df["id"] == user_id]["movie_id"].values)
    recs = [mid for mid in global_pop_list if mid not in seen]
    return recs[:n_recs]


print("Function defined: get_pop_recommendations")

# Kiểm tra hợp lý
sample_user = train_df["id"].iloc[0]
sample_recs = get_pop_recommendations(sample_user, train_df, global_pop_list, n_recs=5)
print(f"\nSample recs for user {sample_user}: {sample_recs}")
sample_scores = [popularity_score[m] for m in sample_recs]
print(f"Popularity counts             : {sample_scores}")

Function defined: get_pop_recommendations

Sample recs for user 103007: [173696, 531662, 800220, 519065, 186830]
Popularity counts             : [np.int64(288), np.int64(255), np.int64(249), np.int64(243), np.int64(237)]


## 3. Thực nghiệm và Kết quả (Experiments & Results)

### 3.1 Các hàm đánh giá
Các hàm đánh giá tính Precision@$K$, Recall@$K$, NDCG@$K$, HR@$K$ và MRR@$K$ trên các tương tác dương được giữ lại. Quy trình mask item train được áp dụng để tránh đề xuất lại các item đã biết.


In [52]:
# [Giải thích cell]
# - Mục đích: định nghĩa hàm đánh giá Top-K trên các held-out interactions dương.
# - Đầu vào chính: ma trận held-out sau threshold, `train_interactions` để mask lịch sử và mô hình sinh điểm.
# - Kết quả tạo ra: Precision@K, Recall@K, NDCG@K, HR@K và MRR@K.
# - Lưu ý: validation/test chỉ được dùng để đo hiệu năng; không cập nhật trọng số mô hình.

def evaluate_popularity(
    test_split_df,
    train_df,
    global_pop_list,
    k_values=(5, 10, 20, 50),
):
    """
    Tính Precision@K, Recall@K, NDCG@K, HR@K, MRR@K cho Popularity Baseline.

    Parameters
    ----------
    test_split_df  : DataFrame với cột ['id', 'movie_id', 'rate']
    train_df       : DataFrame train (để loại trừ items đã xem)
    global_pop_list: list of movie_ids sorted by descending popularity
    k_values       : tuple of K values

    Returns
    -------
    dict {K: {metric: value}}
    """
    if test_split_df is None or len(test_split_df) == 0:
        return {k: {"precision": 0., "recall": 0., "ndcg": 0.,
                     "hr": 0., "mrr": 0.} for k in k_values}

    max_k = max(k_values)
    accum = {k: {"precision": [], "recall": [], "ndcg": [],
                  "hr": [], "mrr": []} for k in k_values}

    # Chỉ evaluate users có positive test items (rate >= POSITIVE_THRESHOLD)
    test_pos = test_split_df[test_split_df["rate"] >= POSITIVE_THRESHOLD]

    # Nhóm ground truth theo người dùng
    user_gt = test_pos.groupby("id")["movie_id"].apply(set).to_dict()

    for user_id, true_items in user_gt.items():
        if not true_items:
            continue

        # Lấy top-max_k recommendations
        recs = get_pop_recommendations(user_id, train_df, global_pop_list, n_recs=max_k)

        n_relevant = len(true_items)
        relevance  = np.array([1.0 if mid in true_items else 0.0 for mid in recs])

        # Đệm danh sách nếu số item nhỏ hơn max_k (rất hiếm, nhưng an toàn)
        if len(relevance) < max_k:
            relevance = np.pad(relevance, (0, max_k - len(relevance)))

        for k in k_values:
            rel_k  = relevance[:k]
            n_hits = float(rel_k.sum())

            precision = n_hits / k
            recall    = n_hits / n_relevant if n_relevant > 0 else 0.0
            hr        = 1.0 if n_hits > 0 else 0.0

            discounts = 1.0 / np.log2(np.arange(2, k + 2))
            dcg  = float((rel_k * discounts).sum())
            idcg = float(discounts[:min(n_relevant, k)].sum())
            ndcg = dcg / idcg if idcg > 0 else 0.0

            mrr = 0.0
            for j in range(k):
                if rel_k[j] > 0:
                    mrr = 1.0 / (j + 1)
                    break

            accum[k]["precision"].append(precision)
            accum[k]["recall"].append(recall)
            accum[k]["ndcg"].append(ndcg)
            accum[k]["hr"].append(hr)
            accum[k]["mrr"].append(mrr)

    summary = {}
    for k in k_values:
        summary[k] = {m: float(np.mean(v)) if v else 0.0
                      for m, v in accum[k].items()}
        summary[k]["n_users"] = len(accum[k]["precision"])
    return summary


def print_metrics(metrics, label):
    n_users = list(metrics.values())[0].get("n_users", "?")
    print(f"\n=== {label} (n_users={n_users}) ===")
    print(f"{'K':>4} | {'Prec':>8} | {'Recall':>8} | {'NDCG':>8} | {'HR':>8} | {'MRR':>8}")
    print("-" * 58)
    for k, m in sorted(metrics.items()):
        print(f"{k:>4} | {m['precision']:>8.4f} | {m['recall']:>8.4f} | "
              f"{m['ndcg']:>8.4f} | {m['hr']:>8.4f} | {m['mrr']:>8.4f}")


print("Evaluation functions defined: evaluate_popularity, print_metrics")

Evaluation functions defined: evaluate_popularity, print_metrics


### 3.2 Đánh giá trên tập validation
Cell bên dưới báo cáo hiệu năng trên tập validation. Bảng 1 trình bày các chỉ số Top-$K$ của Popularity Baseline, đóng vai trò kiểm tra nhất quán trước khi đánh giá trên tập test.


In [53]:
# [Giải thích cell]
# - Mục đích: báo cáo metric trên tập validation sau khi mô hình/hyperparameter đã sẵn sàng.
# - Đầu vào chính: tập validation và các artifact cần thiết để sinh ranking.
# - Kết quả tạo ra: `valid_metrics`, dùng để kiểm tra hiệu năng trước khi đánh giá test.

print("Evaluating on VALID set...")
valid_metrics = evaluate_popularity(
    test_split_df=valid_df,
    train_df=train_df,
    global_pop_list=global_pop_list,
    k_values=K_VALUES,
)
print_metrics(valid_metrics, "VALID")

Evaluating on VALID set...

=== VALID (n_users=240) ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0017 |   0.0083 |   0.0060 |   0.0083 |   0.0052
  10 |   0.0017 |   0.0167 |   0.0086 |   0.0167 |   0.0063
  20 |   0.0013 |   0.0250 |   0.0106 |   0.0250 |   0.0068
  50 |   0.0016 |   0.0792 |   0.0214 |   0.0792 |   0.0085


### 3.3 Đánh giá trên tập kiểm thử
Hiệu năng cuối được báo cáo trên ba phân tách: toàn bộ test, test-warm và test-cold. Bảng 2 giúp quan sát trực tiếp giới hạn của phương pháp độ phổ biến trong tình huống item chưa từng xuất hiện ở train.


In [54]:
# [Giải thích cell]
# - Mục đích: chạy đánh giá cuối cùng trên các split validation/test đã xác định.
# - Đầu vào chính: mô hình đã huấn luyện, train interactions và các tập held-out.
# - Kết quả tạo ra: bảng metric cho full, warm và cold split nếu có.
# - Lưu ý: không dùng kết quả test để chọn siêu tham số.

print("=" * 65)
print("FINAL EVALUATION — POPULARITY BASELINE")
print(f"Cấu hình: threshold={POSITIVE_THRESHOLD}, popularity=train_positive_count")
print("=" * 65)

eval_sets = [
    ("VALID",     valid_df),
    ("TEST",      test_df),
    ("TEST_WARM", test_warm_df),
    ("TEST_COLD", test_cold_df),
]

all_results = {}
for label, split_df in eval_sets:
    metrics = evaluate_popularity(
        test_split_df=split_df,
        train_df=train_df,
        global_pop_list=global_pop_list,
        k_values=K_VALUES,
    )
    all_results[label] = metrics
    print_metrics(metrics, label)


FINAL EVALUATION — POPULARITY BASELINE
Config: threshold=7, popularity=train_positive_count

=== VALID (n_users=240) ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0017 |   0.0083 |   0.0060 |   0.0083 |   0.0052
  10 |   0.0017 |   0.0167 |   0.0086 |   0.0167 |   0.0063
  20 |   0.0013 |   0.0250 |   0.0106 |   0.0250 |   0.0068
  50 |   0.0016 |   0.0792 |   0.0214 |   0.0792 |   0.0085

=== TEST (n_users=262) ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0023 |   0.0115 |   0.0115 |   0.0115 |   0.0115
  10 |   0.0019 |   0.0191 |   0.0141 |   0.0191 |   0.0126
  20 |   0.0011 |   0.0229 |   0.0150 |   0.0229 |   0.0128
  50 |   0.0012 |   0.0611 |   0.0221 |   0.0611 |   0.0138

=== TEST_WARM (n_users=253) ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------

### 3.4 Tổng hợp kết quả
Các chỉ số được gom về một `DataFrame` để thuận tiện cho việc so sánh với LightFM Baseline, ItemKNN và SeRel-LightFM. Bảng 3 là bảng kết quả tổng hợp được xuất bởi cell bên dưới.


In [55]:
# [Giải thích cell]
# - Mục đích: chạy đánh giá cuối cùng trên các split validation/test đã xác định.
# - Đầu vào chính: mô hình đã huấn luyện, train interactions và các tập held-out.
# - Kết quả tạo ra: bảng metric cho full, warm và cold split nếu có.
# - Lưu ý: không dùng kết quả test để chọn siêu tham số.

rows = []
for split_label, metrics in all_results.items():
    for k, m in sorted(metrics.items()):
        rows.append({
            "Split"    : split_label,
            "K"        : k,
            "Precision": round(m["precision"], 4),
            "Recall"   : round(m["recall"],    4),
            "NDCG"     : round(m["ndcg"],      4),
            "HR"       : round(m["hr"],        4),
            "MRR"      : round(m["mrr"],       4),
            "N_users"  : m.get("n_users", None),
        })

results_df = pd.DataFrame(rows)
display(results_df)

# Xuất kết quả ra CSV
results_df.to_csv("popularity_baseline_results.csv", index=False)
print("\nSaved: popularity_baseline_results.csv")

,Split,K,Precision,Recall,NDCG,HR,MRR,N_users
0,VALID,5,0.0017,0.0083,0.0060,0.0083,0.0052,240
1,VALID,10,0.0017,0.0167,0.0086,0.0167,0.0063,240
2,VALID,20,0.0013,0.0250,0.0106,0.0250,0.0068,240
3,VALID,50,0.0016,0.0792,0.0214,0.0792,0.0085,240
4,TEST,5,0.0023,0.0115,0.0115,0.0115,0.0115,262
5,TEST,10,0.0019,0.0191,0.0141,0.0191,0.0126,262
6,TEST,20,0.0011,0.0229,0.0150,0.0229,0.0128,262
7,TEST,50,0.0012,0.0611,0.0221,0.0611,0.0138,262
8,TEST_WARM,5,0.0024,0.0119,0.0119,0.0119,0.0119,253
9,TEST_WARM,10,0.0020,0.0198,0.0146,0.0198,0.0131,253



Saved: popularity_baseline_results.csv


### 3.5 Minh họa gợi ý cho một người dùng
Cell này minh họa danh sách top-$N$ item được đề xuất cho một người dùng cụ thể sau khi loại bỏ item đã thấy trong tập huấn luyện. Bảng 4 là ví dụ định tính cho hành vi của mô hình độ phổ biến.


In [56]:
# [Giải thích cell]
# - Mục đích: minh họa quy trình suy luận top-N cho một người dùng cụ thể.
# - Đầu vào chính: user_id, mô hình hoặc danh sách popularity, metadata phim và train history.
# - Kết quả tạo ra: bảng phim được đề xuất sau khi loại bỏ item người dùng đã thấy.
# - Lưu ý: đây là ví dụ định tính, không thay thế cho đánh giá định lượng Top-K.

def recommend_for_user_pop(user_id, train_df, global_pop_list,
                            movies_df, n_recs=10):
    """Hiển thị top-N recommendations theo popularity cho 1 user."""
    recs = get_pop_recommendations(user_id, train_df, global_pop_list, n_recs=n_recs)
    result = movies_df[movies_df["id"].isin(recs)][
        ["id", "original_title", "genres", "year_published", "rate"]
    ].copy()
    result["pop_count"] = result["id"].map(popularity_score)
    result["pop_rank"]  = result["id"].map(popularity_rank) + 1  # 1-indexed
    # Sắp xếp theo popularity
    result = result.sort_values("pop_rank")
    return result


sample_user = train_df["id"].iloc[0]
print(f"[Popularity Baseline] Top-10 gợi ý cho user_id = {sample_user}\n")
recs = recommend_for_user_pop(
    user_id=sample_user,
    train_df=train_df,
    global_pop_list=global_pop_list,
    movies_df=movies_df,
    n_recs=10,
)
print(recs.to_string(index=False))

[Popularity Baseline] Top-10 gợi ý cho user_id = 103007

    id       original_title                                          genres  year_published  rate  pop_count  pop_rank
173696       Shutter Island                                Thriller|Intriga          2010.0   7.6        288        63
531662                Tesis                                Intriga|Thriller          1996.0   7.7        255       127
800220              Aladdin Animación|Fantástico|Musical|Infantil|Aventuras          1992.0   7.4        249       136
519065         The Exorcist                                          Terror          1973.0   7.6        243       147
186830                  300              Acción|Bélico|Aventuras|Fantástico          2006.0   7.2        237       156
779937         Nightcrawler                                        Thriller          2014.0   7.3        232       162
966887 Beauty and the Beast   Animación|Fantástico|Romance|Musical|Infantil          1991.0   7.3        231  

## 4. Kết luận (Conclusion)
Popularity Baseline cung cấp mốc so sánh tối thiểu cho hệ gợi ý phim. Do chỉ dựa vào số lượng tương tác dương toàn cục, mô hình thường ổn định trên item warm nhưng bị hạn chế ở item cold và không thể cá nhân hóa sâu theo sở thích từng người dùng. Kết quả của notebook này nên được diễn giải như ngưỡng tham chiếu khi đánh giá các mô hình có học tham số và có sử dụng đặc trưng phụ.
